In [1]:
# ============================================================
# CELDA 1 · Cargar datos y definir las columnas por tipo
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import joblib
import os

# Cargar el dataset original
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

# ── Ver los 11 valores problemáticos de TotalCharges ────────

# Convertir a numérico 
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Mostrar las filas donde TotalCharges es NaN
filas_problematicas = df[df['TotalCharges'].isna()]

print(f"\nFilas con TotalCharges vacío: {len(filas_problematicas)}")
print("\n=== LOS 11 CLIENTES PROBLEMÁTICOS ===")
print(filas_problematicas[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].to_string())

# ── Corrección obligatoria detectada en el EDA ──────────────
# TotalCharges tiene 11 espacios vacíos que se corresponden con una antigüedad de cero → convertir a número
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Rellenar los 11 nulos resultantes con 0 (clientes con tenure=0)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# ── Eliminar customerID (identificador, no predice nada) ────
df = df.drop(columns=['customerID'])

# ── Convertir la variable objetivo a número ─────────────────
# El modelo necesita 0 y 1, no "Yes" y "No"
# El 1 significa que el cliente se va
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f"\nDistribución Churn después de la conversión:")
print(df['Churn'].value_counts())

# ── Definir columnas por tipo ────────────────────────────────
# Estas listas le dicen al pipeline qué transformación aplicar a cada columna
# Las numéricas recibirán escalado, las categóricas recibirán codificación. 
# Es importante definirlas explícitamente porque cada tipo necesita un tratamiento diferente.

columnas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

columnas_categoricas = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

print(f"\nColumnas numéricas: {columnas_numericas}")
print(f"\nColumnas categóricas: {columnas_categoricas}")
print(f"\nTotal columnas a usar: {len(columnas_numericas) + len(columnas_categoricas)}")

Dataset cargado: 7043 filas x 21 columnas

Filas con TotalCharges vacío: 11

=== LOS 11 CLIENTES PROBLEMÁTICOS ===
      tenure  MonthlyCharges  TotalCharges Churn
488        0           52.55           NaN    No
753        0           20.25           NaN    No
936        0           80.85           NaN    No
1082       0           25.75           NaN    No
1340       0           56.05           NaN    No
3331       0           19.85           NaN    No
3826       0           25.35           NaN    No
4380       0           20.00           NaN    No
5218       0           19.70           NaN    No
6670       0           73.35           NaN    No
6754       0           61.90           NaN    No

Distribución Churn después de la conversión:
Churn
0    5174
1    1869
Name: count, dtype: int64

Columnas numéricas: ['tenure', 'MonthlyCharges', 'TotalCharges']

Columnas categóricas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Onli

In [2]:
# ============================================================
# CELDA 2 · Construir el ColumnTransformer (el pipeline)
# ============================================================

# Se define la lógica de transformación

# ── Pipeline para columnas NUMÉRICAS ────────────────────────
# Paso 1: SimpleImputer → rellena nulos con la mediana (por si hubiera más en producción)
# Paso 2: StandardScaler → escala los valores para que tengan media=0 y std=1
#         Ejemplo: tenure va de 0-72, MonthlyCharges de 18-119
#         Después del escalado ambas estarán en el mismo rango → el modelo las trata igual
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# ── Pipeline para columnas CATEGÓRICAS ──────────────────────
# Paso 1: SimpleImputer → rellena nulos con el valor más frecuente (por si hubiera en producción)
# Paso 2: OneHotEncoder → convierte texto a columnas 0/1. Los modelos no entienden texto, 
# así que convierte cada categoría en una columna nueva de 0s y 1s.
#         Ejemplo: Contract tiene 3 valores → genera 3 columnas nuevas
#         Month-to-month → [1, 0, 0]
#         One year       → [0, 1, 0]
#         Two year       → [0, 0, 1]
#         handle_unknown='ignore' → si llega una categoría nueva de contrato la codifica como todo ceros
#         sparse_output=False → hace que el resultado sea una matriz densa (Numpy array) en vez de una matriz dispersa
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── ColumnTransformer · combina ambos pipelines ──────────────
# Aplica cada pipeline solo a las columnas que le corresponden
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, columnas_numericas),
    ('cat', categorical_pipeline, columnas_categoricas)
])

print("✅ Pipeline de preprocesamiento definido correctamente")
print("\nEstructura del pipeline:")
print(preprocessor)

✅ Pipeline de preprocesamiento definido correctamente

Estructura del pipeline:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'Multi

In [3]:
# ============================================================
# CELDA 3 · Train/Test split y ajuste del pipeline
# ============================================================

# Separar features (X) de la variable objetivo (y)
X = df.drop(columns=['Churn'])   # Todo menos Churn
y = df['Churn']                  # Solo Churn

print(f"X shape: {X.shape}")     # Debería ser (7043, 19)
print(f"y shape: {y.shape}")     # Debería ser (7043,)

# ── Train/Test split ─────────────────────────────────────────
# 80% para entrenar el modelo, 20% para evaluarlo
# stratify=y → mantiene la proporción 73/27 en ambos conjuntos
# random_state=42 → hace el split reproducible (siempre el mismo resultado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,  # el split será siempre igual se ejecute donde se ejecute
    stratify=y        # importante con datasets desbalanceados: mantiene la proporción
)                     # 73/27 original en Churn, tanto en train como en test   

print(f"\nTamaños después del split:")
print(f"  X_train: {X_train.shape}  →  {X_train.shape[0]} clientes para entrenar")
print(f"  X_test:  {X_test.shape}   →  {X_test.shape[0]} clientes para evaluar")

print(f"\nDistribución Churn en train:")
print(y_train.value_counts(normalize=True).mul(100).round(1))
print(f"\nDistribución Churn en test:")
print(y_test.value_counts(normalize=True).mul(100).round(1))

# ── Ajustar el pipeline SOLO con los datos de entrenamiento ──
# IMPORTANTE: nunca ajustar el pipeline con X_test
# Si lo hicieras, el modelo "vería" los datos de test antes de evaluarse
# → las métricas serían falsamente optimistas (data leakage)
preprocessor.fit(X_train)

print("\n✅ Pipeline ajustado con los datos de entrenamiento")

X shape: (7043, 19)
y shape: (7043,)

Tamaños después del split:
  X_train: (5634, 19)  →  5634 clientes para entrenar
  X_test:  (1409, 19)   →  1409 clientes para evaluar

Distribución Churn en train:
Churn
0    73.5
1    26.5
Name: proportion, dtype: float64

Distribución Churn en test:
Churn
0    73.5
1    26.5
Name: proportion, dtype: float64

✅ Pipeline ajustado con los datos de entrenamiento


In [4]:
# ============================================================
# CELDA 4 · Transformar los datos y guardar todo
# ============================================================

# Aplicar las transformaciones. El resultado es una matriz de números puros
# OneHotEncoder expande las columnas categóricas
# Aquí sí se transforma X_test 
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Shape ANTES del preprocesamiento: {X_train.shape}")
print(f"Shape DESPUÉS del preprocesamiento: {X_train_processed.shape}")
print(f"\n→ De 19 columnas originales a {X_train_processed.shape[1]} columnas")
print(f"  (las categóricas se expandieron con OneHotEncoding)")

# ── Guardar los datos procesados en el formato nativo de Numpy ─────────────────────────────
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train_processed)
np.save('../data/processed/X_test.npy',  X_test_processed)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy',  y_test.values)

print("\n✅ Datos procesados guardados en data/processed/")

# ── Guardar el pipeline ───────────────────────────────────────
os.makedirs('../models', exist_ok=True)
# Serializa el pipeline completo en un archivo .pkl (pickle). 
# Serializar significa convertir un objeto Python en un archivo que puedes guardar en disco y cargar después.
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print("✅ Pipeline guardado en models/preprocessor.pkl")

# ── Verificación final ────────────────────────────────────────
print("\n=== VERIFICACIÓN FINAL ===")
print(f"preprocessor.pkl existe: {os.path.exists('../models/preprocessor.pkl')}")
print(f"X_train.npy existe:      {os.path.exists('../data/processed/X_train.npy')}")
print(f"X_test.npy existe:       {os.path.exists('../data/processed/X_test.npy')}")

Shape ANTES del preprocesamiento: (5634, 19)
Shape DESPUÉS del preprocesamiento: (5634, 46)

→ De 19 columnas originales a 46 columnas
  (las categóricas se expandieron con OneHotEncoding)

✅ Datos procesados guardados en data/processed/
✅ Pipeline guardado en models/preprocessor.pkl

=== VERIFICACIÓN FINAL ===
preprocessor.pkl existe: True
X_train.npy existe:      True
X_test.npy existe:       True


In [5]:
# ============================================================
# CELDA 5 · Simular un cliente nuevo pasando por el pipeline
# ============================================================

# Cliente de prueba con datos reales
cliente_nuevo = pd.DataFrame([{
    'gender': 'Male',
    'SeniorCitizen': '0',
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 5,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.0,
    'TotalCharges': 475.0
}])

# Transformar con el pipeline guardado
cliente_procesado = preprocessor.transform(cliente_nuevo)

print(f"Cliente original:    {cliente_nuevo.shape[1]} columnas")
print(f"Cliente procesado:   {cliente_procesado.shape[1]} columnas")
print(f"\n→ Listo para pasar al modelo")
print(f"\nPerfil del cliente de prueba:")
print(f"  Contrato mes a mes + Fiber optic + sin servicios de seguridad")
print(f"  + pago por cheque electrónico + solo lleva 5 meses")
print(f"  → Según el EDA, este cliente tiene perfil de ALTO riesgo de churn")

Cliente original:    19 columnas
Cliente procesado:   46 columnas

→ Listo para pasar al modelo

Perfil del cliente de prueba:
  Contrato mes a mes + Fiber optic + sin servicios de seguridad
  + pago por cheque electrónico + solo lleva 5 meses
  → Según el EDA, este cliente tiene perfil de ALTO riesgo de churn


# CONCLUSIONES #  

**Datos limpios**:

Se encontraron exactamente 11 valores problemáticos en TotalCharges, todos con tenure = 0 (clientes nuevos sin cargo total aún). Se rellenaron con 0, que es la decisión correcta.
El dataset quedó con 7.043 filas limpias, sin nulos reales.

**Desbalanceo confirmado**:

73.5% no churn / 26.5% churn — el split stratify=y preservó esta proporción exactamente igual en train y test, lo cual es importante para que las métricas sean honestas.

**Pipeline construido correctamente**:

3 columnas numéricas → SimpleImputer (mediana) + StandardScaler
16 columnas categóricas → SimpleImputer (moda) + OneHotEncoder
El pipeline se entrenó solo con X_train (sin data leakage ✅)

**Expansión de columnas**:

De 19 columnas originales → 46 columnas tras el OneHotEncoding. Esto es normal y esperado.

**Archivos guardados correctamente**:

models/preprocessor.pkl ✅
data/processed/X_train.npy, X_test.npy, y_train.npy, y_test.npy ✅

**Verificación con cliente "real"**:

El pipeline transforma correctamente un cliente nuevo de 19 → 46 columnas. El cliente de prueba (contrato mes a mes + Fiber optic + 5 meses de antigüedad) tiene perfil de alto riesgo de churn, consistente con el EDA.